<a href="https://colab.research.google.com/github/lamaljalal/NLP/blob/main/Lab5_Text_Representation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Text Representation

Text representation in NLP means converting text into a numerical form that a machine learning model can understand and process.

## What is Embedding?

Embeddings in NLP is a technique where individual words are represented as real-valued vectors and captures inter-word semantics.


In this notebook, We going to intreduce 2 techniques for embedding. These techniques will be used for a machine learning models such as SVM, Random forest, ... ect.

<img src="https://drive.google.com/uc?export=view&id=1jd0u_sGppDKqBYbhtJfGxp14lnUWUTTw" width="900">

# 1- TF-IDF

<img src="https://drive.google.com/uc?export=view&id=1saSt0mMgOQ2ybbTms1lu_ruhUcBWkKzL" width="500">

TF-IDF stands for term frequency-inverse document frequency. It is a measure that discounts common words. Used in the fields of information retrieval (IR) and machine learning, that can quantify the importance of words in a document amongst a collection of documents (also known as a corpus).

### Components of TF-IDF

1. **TF (Term Frequency):**  
   Measures how frequently a term appears in a document.


$$ \text{TF}(t, d) = \frac{\text{Number of times term } t \text{ appears in document } d}{\text{Total number of terms in document } d} $$


2. **IDF (Inverse Document Frequency):**  
   Measures how important a term is across all documents. Words that appear in many documents get lower scores.

$$    \text{IDF}(t) = \log \left(\frac{\text{Number of all documents N}}{\text{Number of documents containing the term } t}\right) $$

<img src="https://drive.google.com/uc?export=view&id=1JqPILC8TTh3yDQZCSuPNXwm6ER5YeJFh" width="900">

In [1]:
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer


In [2]:
doc_1 = "Data is the oil of the digital economy"
doc_2 = "Data is a new oil"

data = [doc_1, doc_2]


In [3]:
tfidf = TfidfVectorizer()
result = tfidf.fit_transform(data) # returns sparce matrix

In [4]:
df = pd.DataFrame(result.toarray(), columns=tfidf.get_feature_names_out())
df

,data,digital,economy,is,new,of,oil,the
0,0.243777,0.34262,0.34262,0.243777,0.000000,0.34262,0.243777,0.68524
1,0.448321,0.00000,0.00000,0.448321,0.630099,0.00000,0.448321,0.00000


# Cosine similarity

In NLP, Cosine similarity is a metric used to measure how similar the documents are.


$$ \text{cosine similarity} = \frac{A \cdot B}{\|A\| \times \|B\|} $$

Where:  
$ A \cdot B $  = dot product of vectors A and B  
$ \|A\| $ = magnitude (length) of vector A  
$ \|B\| $ =  magnitude (length) of vector B

#### Intuition

- If vectors point in the **same direction**, cosine similarity = **1** (maximum similarity).
- If vectors are **orthogonal (90° apart)**, cosine similarity = **0** (no similarity).

<img src="https://drive.google.com/uc?export=view&id=1b-o8CVfHBsjUGY_hXmxevU91gHidIuxO" width="900">

In [5]:
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

In [6]:
# Sample text data (replace with your own documents)

doc_1 = "Data is the oil of the digital economy"
doc_2 = "Data is a new oil"

data = [doc_1, doc_2]

In [7]:
tfidf_vectorizer = TfidfVectorizer() # Create a CountVectorizer instance
vector_matrix = tfidf_vectorizer.fit_transform(data) # Fit and transform the documents into numerical vectors

In [8]:
# Calculate the cosine similarity between the documents
cosine_similarity_matrix = cosine_similarity(vector_matrix)

df_cosine = pd.DataFrame(data=cosine_similarity_matrix, index=data, columns=data)

df_cosine

,Data is the oil of the digital economy,Data is a new oil
Data is the oil of the digital economy,1.000000,0.327871
Data is a new oil,0.327871,1.000000


# 2- What is Word2vec?

Word2Vec consists of models for generating word embedding. These models are two-layer neural networks having one input layer, one hidden layer, and one output layer.

Word2Vec utilizes two architectures :
1. CBOW (Continuous Bag of Words)
2. **Skip Gram**
<img src="https://drive.google.com/uc?export=view&id=1K38nEu_KhgtSJuG2RySwc6U5AAjVAgWZ" width="600">


Run this command in terminal to install
> pip install gensim

We will use fake and real news dataset to do our expirament. You can find the dataset here: https://www.kaggle.com/datasets/clmentbisaillon/fake-and-real-news-dataset?select=True.csv


In [10]:
!pip install -q gensim
import pandas as pd
import nltk
import numpy as np
import gensim

from nltk.tokenize import word_tokenize

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 27.8/27.8 MB 46.5 MB/s eta 0:00:00


In [11]:
from pathlib import Path
true_path = Path('/content/True.csv')
if true_path.exists():
    df = pd.read_csv(true_path, engine='python', on_bad_lines='skip')
    display(df.head())
else:
    df = None
    print('True.csv not provided; skipping the optional news Word2Vec demo.')


,title,text,subject,date
0,"As U.S. budget fight looms, Republicans flip t...",WASHINGTON (Reuters) - The head of a conservat...,politicsNews,"December 31, 2017"
1,U.S. military to accept transgender recruits o...,WASHINGTON (Reuters) - Transgender people will...,politicsNews,"December 29, 2017"
2,Senior U.S. Republican senator: 'Let Mr. Muell...,WASHINGTON (Reuters) - The special counsel inv...,politicsNews,"December 31, 2017"
3,FBI Russia probe helped by Australian diplomat...,WASHINGTON (Reuters) - Trump campaign adviser ...,politicsNews,"December 30, 2017"
4,Trump wants Postal Service to charge 'much mor...,SEATTLE/WASHINGTON (Reuters) - President Donal...,politicsNews,"December 29, 2017"


In [12]:
tokens = []
if df is not None:
    for i in df['text'].dropna():
        token = i.split()
        tokens.append(token)


In [13]:
if tokens:
    w2v = gensim.models.Word2Vec(tokens, min_count=1, vector_size=100, window=5, sg=1)
else:
    w2v = None


In [14]:
if w2v is not None and all(w in w2v.wv for w in ['provide', 'program']):
    print("Cosine similarity between 'provide' and 'program' - Skip Gram : ", w2v.wv.similarity('provide', 'program'))
else:
    print('Optional news demo skipped.')


Cosine similarity between 'provide' and 'program' - Skip Gram :  0.9408097


In [15]:
if w2v is not None and 'program' in w2v.wv:
    print("words that similar to 'program' - Skip Gram : ", w2v.wv.most_similar('program'))
else:
    print('Optional news demo skipped.')


words that similar to 'program' - Skip Gram :  [('prejudice', 0.9869276881217957), ('doesn’t', 0.9864057898521423), ('present', 0.9850351810455322), ('file', 0.9850038886070251), ('undermine', 0.9847455620765686), ('important', 0.9844566583633423), ('looks', 0.9842040538787842), ('ease', 0.9842008948326111), ('argue', 0.984032154083252), ('needs', 0.9835244417190552)]


# Tasks

### Task 1: Cosine Similarity
Use the Cosine Similarity method to determine how similar the following sentences are.

'This is the first document.',  
'This document is the second document.',  
'And this is the third one.',  
'Is this the first document?'  


In [16]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
import pandas as pd

sentences = [
    'This is the first document.',
    'This document is the second document.',
    'And this is the third one.',
    'Is this the first document?'
]

vectorizer = TfidfVectorizer()
tfidf_matrix = vectorizer.fit_transform(sentences)
similarity_matrix = cosine_similarity(tfidf_matrix)

cosine_df = pd.DataFrame(
    similarity_matrix,
    index=[f'Sentence {i}' for i in range(1, 5)],
    columns=[f'Sentence {i}' for i in range(1, 5)]
)
cosine_df


,Sentence 1,Sentence 2,Sentence 3,Sentence 4
Sentence 1,1.000000,0.646926,0.307772,1.000000
Sentence 2,0.646926,1.000000,0.225240,0.646926
Sentence 3,0.307772,0.225240,1.000000,0.307772
Sentence 4,1.000000,0.646926,0.307772,1.000000


### Task 2: TF-IDF
Use tf-idf method on the sentences below to determine the important words.

'data science is one of the most important fields of science',  
'this is one of the best data science courses',  
'data scientists analyze data'  


In [17]:
from sklearn.feature_extraction.text import TfidfVectorizer
import pandas as pd

sentences_tfidf = [
    'data science is one of the most important fields of science',
    'this is one of the best data science courses',
    'data scientists analyze data'
]

tfidf_vectorizer = TfidfVectorizer()
tfidf_values = tfidf_vectorizer.fit_transform(sentences_tfidf)

tfidf_df = pd.DataFrame(
    tfidf_values.toarray(),
    columns=tfidf_vectorizer.get_feature_names_out(),
    index=[f'Sentence {i}' for i in range(1, 4)]
)
display(tfidf_df)

print('Most important words in each sentence:')
for i, row in tfidf_df.iterrows():
    top_words = row[row > 0].sort_values(ascending=False)
    print(f'{i}:', ', '.join(f'{word} ({score:.3f})' for word, score in top_words.items()))


,analyze,best,courses,data,fields,important,is,most,of,one,science,scientists,the,this
Sentence 1,0.000000,0.000000,0.000000,0.189526,0.320895,0.320895,0.244049,0.320895,0.488098,0.244049,0.488098,0.000000,0.244049,0.000000
Sentence 2,0.000000,0.400294,0.400294,0.236420,0.000000,0.000000,0.304434,0.000000,0.304434,0.304434,0.304434,0.000000,0.304434,0.400294
Sentence 3,0.542701,0.000000,0.000000,0.641055,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.542701,0.000000,0.000000


Most important words in each sentence:
Sentence 1: science (0.488), of (0.488), most (0.321), fields (0.321), important (0.321), one (0.244), is (0.244), the (0.244), data (0.190)
Sentence 2: best (0.400), courses (0.400), this (0.400), of (0.304), is (0.304), science (0.304), one (0.304), the (0.304), data (0.236)
Sentence 3: data (0.641), analyze (0.543), scientists (0.543)


## Word2vec

### Task 3:

Download the Simpsons dataset **(simpsons_script_lines.csv)** and apply the preprocessing procedure.  
Use the **'spoken_words'** column.
```
def clean_text(text):
    text = text.lower()
    text = re.sub(r"[0-9]", '', text)
    text = re.sub(r"[)(,”“.’$-]", '', text)
    return text
```
Create a skip gram Word2Vec model as below.
```
Skip_gram_model = gensim.models.Word2Vec(tokens, min_count = 1, vector_size = 100, window = 5, sg = 1)

In [18]:
import pandas as pd
import re
import gensim
from nltk.tokenize import wordpunct_tokenize
from pathlib import Path

# Load the Simpsons dataset (works in the same folder or in Google Colab /content).
csv_path = Path('simpsons_script_lines.csv')
if not csv_path.exists():
    csv_path = Path('/content/simpsons_script_lines.csv')

simpsons_df = pd.read_csv(csv_path, low_memory=False)
simpsons_df = simpsons_df.dropna(subset=['spoken_words']).copy()

def clean_text(text):
    text = text.lower()
    text = re.sub(r"[0-9]", '', text)
    text = re.sub(r"[)(,”“.’$-]", '', text)
    return text

simpsons_df['cleaned_text'] = simpsons_df['spoken_words'].astype(str).apply(clean_text)
tokens = [wordpunct_tokenize(text) for text in simpsons_df['cleaned_text']]

Skip_gram_model = gensim.models.Word2Vec(
    tokens, min_count=1, vector_size=100, window=5, sg=1,
    workers=1, seed=42
)

print('Rows used:', len(simpsons_df))
print('Vocabulary size:', len(Skip_gram_model.wv))
print('Model created successfully.')


Rows used: 43713
Vocabulary size: 23543
Model created successfully.


### Task 4

Use: wv.most_similar() method to :

1.	Find the words similar to “homer”.

2.	Find the words similar to “marge”.

3. Find the words similar to “bart”



In [19]:
for word in ['homer', 'marge', 'bart']:
    print(f"\nWords similar to '{word}':")
    for similar_word, score in Skip_gram_model.wv.most_similar(word, topn=10):
        print(f'{similar_word}: {score:.4f}')



Words similar to 'homer':
abe: 0.8732
bart: 0.8727
mister: 0.8598
marge: 0.8580
j: 0.8546
maggie: 0.8528
ralph: 0.8525
homie: 0.8518
monty: 0.8511
pregnant: 0.8501

Words similar to 'marge':
homie: 0.9095
edna: 0.9024
pregnant: 0.8987
jessica: 0.8932
abe: 0.8925
reverend: 0.8920
sweetie: 0.8913
sweetheart: 0.8908
apu: 0.8887
lenny: 0.8861

Words similar to 'bart':
lisa: 0.9011
maggie: 0.8925
milhouse: 0.8852
homer: 0.8727
nelson: 0.8637
ralph: 0.8563
abe: 0.8557
jessica: 0.8557
homie: 0.8555
apu: 0.8546


### Task 5

Use the wv.doesnt_match() method to :

1.	Find which of 'jimbo', 'milhouse’, and 'kearney’ does not belong to the list.

3.	Find the odd one among "nelson", "bart", and "milhouse".

4.	Find the odd one among ‘homer', 'patty', and ‘selma'.

*Hint: You need to pass the strings as List*

In [20]:
groups = [
    ['jimbo', 'milhouse', 'kearney'],
    ['nelson', 'bart', 'milhouse'],
    ['homer', 'patty', 'selma']
]

for group in groups:
    odd_word = Skip_gram_model.wv.doesnt_match(group)
    print(f'{group} -> odd word: {odd_word}')


['jimbo', 'milhouse', 'kearney'] -> odd word: kearney
['nelson', 'bart', 'milhouse'] -> odd word: bart
['homer', 'patty', 'selma'] -> odd word: homer
